# LOGIKA UI 2026 — ensemble Kumo E043

Notebook ini menghasilkan `application_id,default_probability` dengan 30.000 baris sesuai urutan sample submission.

**Bobot:** Kumo Large 50%, XGBoost 25%, CatBoost 12,5%, LightGBM 12,5%.

**Hasil historis:** pooled OOF KS **0,5151791584**, mean fold KS **0,5176930144**, sample SD **0,0083424910**, AUC **0,8398233966**. Hasil tersebut berasal dari eksperimen eksploratif dan bukan skor hidden test.

Tersedia tiga mode:

- `blend` membangun ulang CSV dari prediksi empat model tersimpan dan memeriksa checksum persis. Default, CPU.
- `inference` menghitung ulang prediksi dari model tree tersimpan serta konteks Kumo. A100/CUDA BF16 diperlukan.
- `train` menjalankan ulang seluruh training/CV dari data mentah lalu membuat ensemble. Proses ini panjang.

Mode `blend` telah dijalankan pada notebook yang disertakan. Jalur inference lengkap dan training ulang lengkap pada notebook ini belum dieksekusi di A100. Training GPU dapat memberi perbedaan numerik meskipun seed sama. Tidak ada pengiriman submission ke Kaggle.

## 1. Persiapan

Jika berada di checkout asli, pilih kernel **Python (Gipta - LOGIKA UI 2026)**.

Di mesin baru, ekstrak `logika-ui-26-e043-reproducible.zip` ke folder baru. ZIP memuat source, notebook, konfigurasi, `uv.lock`, model tree, konteks/feature selection Kumo, dan prediksi OOF/test. Salin data kompetisi ke `data/raw`. Checkpoint Kumo publik tidak disertakan dan akan diunduh dengan revision yang dipin.

Jalankan dari folder hasil ekstraksi pada mesin tempat komputasi dilakukan:

```bash
source scripts/a100_env.sh
uv python install --no-bin 3.11
uv sync --locked --extra gpu --extra kumo --extra cpu_models --inexact
.venv/bin/python -m ipykernel install --user --name logika-ui-26 --display-name "Python (Gipta - LOGIKA UI 2026)"
```

`uv` perlu tersedia. Pilih kernel yang baru terdaftar sebelum menjalankan notebook. Jangan mengubah environment yang sedang dipakai job lain. Detail operasional tersedia di `docs/reproduction/E043.md`.

In [ ]:
from pathlib import Path
import csv
import hashlib
import importlib.metadata
import json
import os
import subprocess
import sys

MODE = "blend"
PROJECT_ROOT = next((path / "artifacts/delivery/e043-project" for path in (Path.cwd(), *Path.cwd().parents) if (path / "artifacts/delivery/e043-project/configs/reproduction/e043.json").is_file()), None)

assert MODE in {"blend", "inference", "train"}
if PROJECT_ROOT is None:
    start = Path.cwd().resolve()
    candidates = [start, *start.parents]
    PROJECT_ROOT = next(
        (path for path in candidates if (path / "configs/reproduction/e043.json").is_file()),
        None,
    )
    if PROJECT_ROOT is None:
        raise FileNotFoundError("Ekstrak ZIP project dan buka notebook dari folder project tersebut")
else:
    PROJECT_ROOT = Path(PROJECT_ROOT).expanduser().resolve()

if sys.version_info[:2] != (3, 11):
    raise RuntimeError("Pilih kernel Python 3.11 dari environment project")

PYTHON = sys.executable
RECIPE = json.loads((PROJECT_ROOT / "configs/reproduction/e043.json").read_text())
OUTPUT = PROJECT_ROOT / "submissions/reproduced_e043" / f"submission_e043_{MODE}.csv"
OUTPUT.parent.mkdir(parents=True, exist_ok=True)
print({"mode": MODE, "python": sys.version.split()[0], "output": str(OUTPUT.relative_to(PROJECT_ROOT))})

## 2. Verifikasi input dan source

Notebook memeriksa identitas sepuluh file mentah terhadap eksperimen asli. Daftar hash terpisah memeriksa source, environment lock, konfigurasi, dan artifact model/prediksi. Notebook boleh diedit untuk memilih mode atau lokasi folder.

In [ ]:
def sha256(path):
    with Path(path).open("rb") as handle:
        return hashlib.file_digest(handle, "sha256").hexdigest()

missing = [name for name in RECIPE["raw_files"] if not (PROJECT_ROOT / "data/raw" / name).is_file()]
if missing:
    raise FileNotFoundError(f"Salin data kompetisi yang belum tersedia ke data/raw: {missing}")

for name, expected in RECIPE["raw_files"].items():
    if sha256(PROJECT_ROOT / "data/raw" / name) != expected:
        raise ValueError(f"Data mentah berbeda dari eksperimen E043: {name}")

manifest = json.loads((PROJECT_ROOT / "configs/reproduction/e043_files.json").read_text())
for name, expected in manifest.items():
    if sha256(PROJECT_ROOT / name) != expected:
        raise ValueError(f"Source atau artifact berubah: {name}")

print({"raw_files_verified": len(RECIPE["raw_files"]), "project_files_verified": len(manifest)})

## 3. Resep dan protokol validasi

- Lima outer fold berstrata **target × bulan keputusan**, seed 2026.
- Inner split 80/20, seed 2026 + nomor fold. Early stopping memakai inner validation, kemudian tree difit ulang pada outer training.
- Feature selection Kumo menggunakan inner training dan inner validation saja. Outer validation dan test tidak boleh masuk konteks.
- Satu aplikasi terpilih per nasabah, pemisahan nasabah train/test, fitur dibatasi pada informasi saat keputusan dibuat. ID dan target tidak menjadi fitur.
- CatBoost menggunakan 633 fitur `credit_v2`, LightGBM/XGBoost menggunakan 683 fitur `credit_v4`. Kumo memilih 120 dari 633 kandidat per fold.
- Kumo Large memakai 16.384 baris konteks, 2 estimator, query batch 256, BF16, tanpa gradient update. Model pretrained dipin ke `nvidia/Kumo-Tabular` revision `v1.0.1`, SDK dipin pada commit di bawah.

Resep ensemble berurutan mempertahankan pembulatan sepuluh desimal CSV di setiap tahap. Hal ini diperlukan untuk kesamaan byte, meskipun bobot efektif dapat ditulis sebagai empat bobot di atas.

In [ ]:
print(json.dumps({
    "weights": RECIPE["weights"],
    "checkpoint": RECIPE["checkpoint"],
    "sdk_commit": RECIPE["sdk_commit"],
    "recipe": "e041 = mean(CatBoost, LightGBM), e042 = mean(e041, XGBoost), e043 = mean(e042, Kumo)",
}, indent=2))

for name, item in RECIPE["bases"].items():
    config = json.loads((PROJECT_ROOT / item["directory"] / "config.json").read_text())
    print(name, json.dumps(config, sort_keys=True))

## 4. Runtime

Mode `blend` memakai CPU. Mode `inference` dan `train` memakai model yang sama dengan eksperimen E043, memerlukan CUDA BF16 dengan setidaknya 9 GiB VRAM bebas. Supervisor membatasi combined RSS worker foundation pada 4 GiB dan menunggu kapasitas RAM akun. Semua model dijalankan berurutan. Pada akun A100 bersama, jangan menjalankan eksperimen lain bersamaan dari notebook ini.

Source worker historis, konteks per fold, feature selection, dan hash checkpoint digunakan kembali saat inference. Cache fitur dibangun dari raw data bila belum tersedia. Tidak ada ketergantungan pada repository privat atau path home penulis.

In [ ]:
packages = ["numpy", "pandas", "duckdb", "pyarrow", "scikit-learn", "lightgbm"]
if MODE != "blend":
    packages += ["catboost", "xgboost-cpu", "torch", "structured-data-models", "huggingface-hub"]
versions = {name: importlib.metadata.version(name) for name in packages}
print(json.dumps(versions, indent=2))

if MODE != "blend":
    gpu_check = """
import json
import torch
if not torch.cuda.is_available() or not torch.cuda.is_bf16_supported():
    raise RuntimeError('CUDA BF16 diperlukan untuk Kumo')
free, total = torch.cuda.mem_get_info()
if free < 9 * 1024**3:
    raise RuntimeError('Butuh setidaknya 9 GiB VRAM bebas')
print(json.dumps({'gpu': torch.cuda.get_device_name(), 'free_vram_gib': free / 1024**3}))
"""
    subprocess.run([PYTHON, "-c", gpu_check], cwd=PROJECT_ROOT, check=True)

## 5. Bangun submission

Dengan `MODE = "blend"`, cell berikut merekonstruksi CSV dari empat prediksi base dan memeriksa hash asli. Dengan `MODE = "inference"`, model dimuat dan seluruh query test dihitung ulang. Dengan `MODE = "train"`, empat base run baru dibuat menggunakan konfigurasi tersimpan, kemudian OOF dan submission ensemble baru disimpan. Kegagalan proses menghasilkan error, bukan fallback ke prediksi tersimpan.

In [ ]:
command = [
    PYTHON,
    str(PROJECT_ROOT / "scripts/reproduce_ensemble.py"),
    "--mode", MODE,
    "--output", str(OUTPUT),
]
subprocess.run(command, cwd=PROJECT_ROOT, check=True)
verification = json.loads(OUTPUT.with_suffix(".verification.json").read_text())
print(json.dumps(verification, indent=2))

## 6. Pemeriksaan format Kaggle

Validasi mencakup nama kolom, jumlah baris, urutan dan keunikan ID, nilai finite, serta rentang probabilitas [0, 1]. File tetap berisi probabilitas, bukan kelas hasil threshold.

In [ ]:
import math

with (PROJECT_ROOT / "data/raw/sample_submission.csv").open(newline="") as handle:
    sample_rows = list(csv.DictReader(handle))
with OUTPUT.open(newline="") as handle:
    reader = csv.DictReader(handle)
    assert reader.fieldnames == ["application_id", "default_probability"]
    rows = list(reader)

ids = [row["application_id"] for row in rows]
values = [float(row["default_probability"]) for row in rows]
assert len(rows) == len(sample_rows) == 30000
assert len(set(ids)) == len(ids)
assert ids == [row["application_id"] for row in sample_rows]
assert all(math.isfinite(value) and 0 <= value <= 1 for value in values)

csv_hash = sha256(OUTPUT)
if MODE == "blend":
    assert csv_hash == RECIPE["submission_sha256"]
print({"rows": len(rows), "minimum_probability": min(values), "maximum_probability": max(values),
       "sha256": csv_hash, "matches_historical_submission": csv_hash == RECIPE["submission_sha256"]})

## 7. Evaluasi OOF

Mode `blend` dan `inference` memeriksa OOF historis E043. Mode `train` menghitung skor dari OOF run baru. Inference test ulang tidak membentuk validasi independen baru. Model dan bobot dipilih setelah beberapa eksperimen pada fold ini, sehingga metrik historis bersifat eksploratif.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score, roc_curve

if MODE == "train":
    oof_path = OUTPUT.with_suffix(".oof.csv")
else:
    oof_path = PROJECT_ROOT / RECIPE["ensemble_directory"] / "oof.csv"
oof = pd.read_csv(oof_path).sort_values("application_id")
train = pd.read_csv(PROJECT_ROOT / "data/raw/train_applications.csv").sort_values("application_id")
assert len(oof) == 70000
assert oof.application_id.is_unique
assert oof.application_id.tolist() == train.application_id.tolist()
assert np.array_equal(oof.target.to_numpy(), train.target.to_numpy())
assert np.isfinite(oof.default_probability).all()
assert oof.default_probability.between(0, 1).all()

fpr, tpr, thresholds = roc_curve(oof.target, oof.default_probability, drop_intermediate=False)
ks = float(np.max(np.abs(tpr - fpr)))
auc = float(roc_auc_score(oof.target, oof.default_probability))
if MODE != "train":
    assert np.isclose(ks, RECIPE["pooled_oof_ks"], rtol=0, atol=1e-14)
print({"oof_rows": len(oof), "pooled_oof_ks": ks, "auc": auc, "oof_scope": "new_training" if MODE == "train" else "historical_exploratory"})

## 8. Output

File pada lokasi yang ditampilkan berikut siap dipilih secara manual di Kaggle. Notebook ini tidak melakukan upload atau submission. Untuk training ulang, identitas empat run berada di file `.training.json`, OOF ensemble berada di `.oof.csv`, dan metrik baru berada di `.metrics.json` di sebelah CSV.

In [ ]:
print(str(OUTPUT))
print("CSV berhasil dibuat dan formatnya tervalidasi")